# Previsão do Tempo com OpenWeatherMap

Este notebook consulta a API do OpenWeatherMap e exibe informações meteorológicas completas com saída em áudio.

## Configuração da API Key (segura)

Sua chave **não deve aparecer no código**. Use um arquivo `.env`:

```bash
# 1. Copie o arquivo de exemplo
cp .env.example .env

# 2. Edite .env e coloque sua chave real
# OPENWEATHER_API_KEY=sua_chave_aqui
```

O arquivo `.env` está no `.gitignore` e **nunca será enviado ao GitHub**.

In [ ]:
import requests
import pyttsx3
import os
from dotenv import load_dotenv

load_dotenv()  # Carrega as variáveis do arquivo .env

API_KEY = os.getenv("OPENWEATHER_API_KEY")
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

if not API_KEY:
    raise ValueError("API key não encontrada. Crie um arquivo .env com OPENWEATHER_API_KEY=sua_chave")

In [ ]:
def obter_previsao(cidade, api_key):
    """Consulta a API e retorna os dados meteorológicos da cidade."""
    params = {
        "q": cidade,
        "appid": api_key,
        "units": "metric",
        "lang": "pt_br"
    }
    return requests.get(BASE_URL, params=params)


def exibir_previsao(cidade, dados):
    """Formata e imprime as informações meteorológicas."""
    main      = dados["main"]
    vento     = dados["wind"]
    descricao = dados["weather"][0]["description"].capitalize()

    temp      = main["temp"]
    sensacao  = main["feels_like"]
    temp_min  = main["temp_min"]
    temp_max  = main["temp_max"]
    umidade   = main["humidity"]
    vel_vento = vento["speed"]

    print(f"\n{'='*45}")
    print(f"  Previsão do Tempo — {cidade.title()}")
    print(f"{'='*45}")
    print(f"  Condição      : {descricao}")
    print(f"  Temperatura   : {temp:.1f}°C")
    print(f"  Sensação term.: {sensacao:.1f}°C")
    print(f"  Mín / Máx     : {temp_min:.1f}°C / {temp_max:.1f}°C")
    print(f"  Umidade       : {umidade}%")
    print(f"  Vento         : {vel_vento} m/s")
    print(f"{'='*45}\n")
    return temp, descricao


def falar(mensagem):
    """Lê a mensagem em voz alta usando pyttsx3."""
    engine = pyttsx3.init()
    engine.setProperty("rate", 150)
    engine.say(mensagem)
    engine.runAndWait()

In [ ]:
cidade = input("Digite a cidade: ").strip()

response = obter_previsao(cidade, API_KEY)

if response.status_code == 200:
    dados = response.json()
    temp, descricao = exibir_previsao(cidade, dados)
    mensagem_audio = (
        f"Em {cidade}, o tempo está {descricao}. "
        f"A temperatura é de {temp:.0f} graus Celsius."
    )
    falar(mensagem_audio)

elif response.status_code == 404:
    msg = f"Cidade '{cidade}' não encontrada. Verifique o nome e tente novamente."
    print(f"Erro: {msg}")
    falar(msg)

elif response.status_code == 401:
    msg = "API key inválida. Verifique sua chave em openweathermap.org."
    print(f"Erro: {msg}")
    falar(msg)

else:
    msg = f"Erro ao consultar a API. Código HTTP: {response.status_code}."
    print(f"Erro: {msg}")
    falar(msg)